# 🔐 Character-Level CNN – Phishing URL Detection  *(v2 – GPU Optimised)*

> **Datasets supported**
> - `begnin_links.csv` — simple `url,label` format (label = 0)
> - `phishing_links.csv` — simple `url,label` format (label = 1)
> - `PhiUSIIL_Phishing_URL_Dataset.csv` — rich feature CSV; only `URL` + `label` columns are used

| Cell | Purpose |
|------|---------|
| 1 | Environment setup & GPU check |
| 2 | **Robust data loading** (handles both CSV schemas) |
| 3 | URL preprocessing & character vocabulary |
| 4 | Character encoding (tokenise → pad) |
| 5 | Stratified train / test split |
| 6 | **Optimised model** (larger embedding, deeper CNN) |
| 7 | Training callbacks |
| 8 | **GPU-optimised training** (batch 512, 50 epochs) |
| 9 | Full evaluation dashboard |
| 10 | Model & artifact export |
| 11 | Inference helper |
| 12 | **Interactive URL Testing Interface** |

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 1 — ENVIRONMENT SETUP
# Mount Drive, create working dir, import libs, set seeds.
# ════════════════════════════════════════════════════════════

import os, random
import numpy as np

# ── Mount Google Drive ───────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

WORK_DIR = '/content/drive/MyDrive/URL_Phishing_Character_Model'
os.makedirs(WORK_DIR, exist_ok=True)
print(f'📁 Working directory : {WORK_DIR}')

# ── Imports ──────────────────────────────────────────────────
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import joblib
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve
)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks, mixed_precision

# ── Seeds ────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

# ── GPU check + mixed-precision (speeds up training ~2×) ─────
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for g in gpus:
        tf.config.experimental.set_memory_growth(g, True)
    mixed_precision.set_global_policy('mixed_float16')
    print(f'⚡ GPU detected: {[g.name for g in gpus]}')
    print('🔀 Mixed-precision (float16) ENABLED')
else:
    print('⚠️  No GPU found – running on CPU (training will be slow)')

print(f'\n🔖 TensorFlow {tf.__version__}')
print('✅ Cell 1 complete.')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 2 — ROBUST DATA LOADING
#
# Handles two CSV schemas automatically:
#
#  Schema A  (simple)           Schema B  (PhiUSIIL rich feature CSV)
#  ──────────────────           ──────────────────────────────────────
#  url,label                    FILENAME,URL,URLLength,...,label
#  https://google.com,0         521848.txt,https://...,31,...,1
#
# In both cases only the URL string and its binary label are kept.
# ════════════════════════════════════════════════════════════

# ── File paths ───────────────────────────────────────────────
# Update these paths if your files live in a different location.
BENIGN_CSV    = os.path.join(WORK_DIR, 'begnin_links.csv')
PHISHING_CSV  = os.path.join(WORK_DIR, 'phishing_links.csv')
PHIUSIIL_CSV  = os.path.join(WORK_DIR, 'PhiUSIIL_Phishing_URL_Dataset.csv')

# ── Smart loader ─────────────────────────────────────────────
def smart_load(path: str) -> pd.DataFrame | None:
    """
    Load any supported CSV and return a clean (url, label) dataframe.

    Detection logic
    ---------------
    • If columns include 'URL' (case-insensitive) → Schema B (PhiUSIIL)
      The 'URL' column contains the URL; 'label' is the binary target.
    • Otherwise → Schema A (simple url,label)
    """
    if not os.path.exists(path):
        print(f'  ⚠️  Not found, skipping: {os.path.basename(path)}')
        return None

    # Read with low_memory=False to avoid dtype inference warnings
    try:
        raw = pd.read_csv(path, encoding='utf-8', low_memory=False)
    except UnicodeDecodeError:
        raw = pd.read_csv(path, encoding='latin-1', low_memory=False)

    # Normalise column names: strip whitespace, lowercase
    raw.columns = [c.strip().lower() for c in raw.columns]

    # ── Schema B: PhiUSIIL or any multi-feature CSV ───────────
    # Recognisable by having a column named 'url' that is NOT the only
    # column (it has many feature columns alongside it).
    if 'url' in raw.columns and 'label' in raw.columns:
        df = raw[['url', 'label']].copy()
        schema = 'Schema B (rich features – extracted url + label)'

    # ── Schema A: plain url,label ─────────────────────────────
    elif set(['url', 'label']).issubset(raw.columns):
        df = raw[['url', 'label']].copy()
        schema = 'Schema A (simple url,label)'

    else:
        print(f'  ❌  Unrecognised schema in {os.path.basename(path)} – columns: {list(raw.columns)}')
        return None

    # Cast label to int and drop NaN rows
    df['label'] = pd.to_numeric(df['label'], errors='coerce')
    df.dropna(subset=['url', 'label'], inplace=True)
    df['label'] = df['label'].astype(int)
    df['url']   = df['url'].astype(str).str.strip()
    df = df[df['url'].str.len() > 0]   # drop blank URLs

    print(f'  📂 {os.path.basename(path):<45} rows: {len(df):>8,}   [{schema}]')
    return df.reset_index(drop=True)


# ── Load all files ────────────────────────────────────────────
print('Loading datasets…')
frames = []
for csv_path in [BENIGN_CSV, PHISHING_CSV, PHIUSIIL_CSV]:
    result = smart_load(csv_path)
    if result is not None:
        frames.append(result)

assert frames, '❌ No CSV files found! Upload at least begnin_links.csv to the WORK_DIR.'

# ── Merge ────────────────────────────────────────────────────
df = pd.concat(frames, ignore_index=True)

# ── Deduplication ────────────────────────────────────────────
before = len(df)
df.drop_duplicates(subset='url', inplace=True)
print(f'\n🗑️  Removed {before - len(df):,} duplicate URLs')

# ── Shuffle ──────────────────────────────────────────────────
df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)

# ── Statistics ───────────────────────────────────────────────
n_total  = len(df)
n_benign = int((df['label'] == 0).sum())
n_phish  = int((df['label'] == 1).sum())

print('\n' + '═'*48)
print('           📊  MERGED DATASET STATISTICS')
print('═'*48)
print(f'  Total samples   : {n_total:>12,}')
print(f'  Benign   (0)    : {n_benign:>12,}   ({100*n_benign/n_total:.1f}%)')
print(f'  Phishing (1)    : {n_phish:>12,}   ({100*n_phish/n_total:.1f}%)')
print('═'*48)

# Class distribution chart
fig, ax = plt.subplots(figsize=(5, 3.5))
bars = ax.bar(['Benign (0)', 'Phishing (1)'], [n_benign, n_phish],
              color=['#4CAF50', '#F44336'], edgecolor='black', width=0.5)
for b in bars:
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + max(n_benign, n_phish)*0.01,
            f'{int(b.get_height()):,}', ha='center', fontsize=11, fontweight='bold')
ax.set_title('Class Distribution', fontsize=13, fontweight='bold')
ax.set_ylabel('Count')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.savefig(os.path.join(WORK_DIR, 'class_distribution.png'), dpi=120)
plt.show()

# Preview
print('\nSample rows:')
display(df.sample(6, random_state=1).reset_index(drop=True))

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 3 — URL PREPROCESSING & CHARACTER VOCABULARY
#
# • Lowercase + strip all URLs
# • Build character-to-index vocabulary from the entire corpus
# • Save artifacts (char2idx, max_length) to Drive via joblib
# ════════════════════════════════════════════════════════════

MAX_LENGTH = 200   # URLs longer than this are truncated; shorter are padded

# ── Step 1: clean text ───────────────────────────────────────
df['url'] = (
    df['url']
    .astype(str)
    .str.lower()
    .str.strip()
)
df = df[df['url'].str.len() > 0].reset_index(drop=True)
print(f'✅ URLs after cleaning: {len(df):,}')

# ── Step 2: build vocabulary ─────────────────────────────────
print('\n⏳ Scanning all URLs to build character vocabulary…')
unique_chars: set = set()
for url in tqdm(df['url'], desc='Vocab scan', ncols=80):
    unique_chars.update(url)

# Sort for determinism; index 0 is reserved for PAD
vocab    = sorted(unique_chars)
char2idx = {ch: i + 1 for i, ch in enumerate(vocab)}   # 1-based
idx2char = {i: ch for ch, i in char2idx.items()}
VOCAB_SIZE = len(char2idx) + 1   # includes PAD token 0

print(f'\n📚 Vocabulary size  : {VOCAB_SIZE}  (including PAD=0)')
print(f'📏 Max URL length   : {MAX_LENGTH}')

# Show a slice of the vocab
print('\nFirst 30 characters in vocabulary:')
print('  ' + '  '.join(f'{repr(c)}={i}' for c, i in list(char2idx.items())[:30]))

# ── Step 3: persist artifacts ────────────────────────────────
ARTIFACTS_PATH = os.path.join(WORK_DIR, 'preprocessing_artifacts.joblib')
ARTIFACTS = dict(
    char2idx    = char2idx,
    idx2char    = idx2char,
    vocab_size  = VOCAB_SIZE,
    max_length  = MAX_LENGTH,
)
joblib.dump(ARTIFACTS, ARTIFACTS_PATH)
print(f'\n💾 Artifacts saved → {ARTIFACTS_PATH}')
print('\n✅ Cell 3 complete.')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 4 — CHARACTER ENCODING
#
# Each URL → list of character indices → zero-padded to MAX_LENGTH
# X shape: (N, MAX_LENGTH)   y shape: (N,)
# ════════════════════════════════════════════════════════════

from tensorflow.keras.preprocessing.sequence import pad_sequences

def encode_url(url: str) -> list:
    """Map each character to its vocab index (0 for unknowns), truncate at MAX_LENGTH."""
    return [char2idx.get(ch, 0) for ch in url[:MAX_LENGTH]]

print('⏳ Encoding URLs…')
sequences = [encode_url(u) for u in tqdm(df['url'], desc='Encoding', ncols=80)]

X = pad_sequences(
    sequences,
    maxlen     = MAX_LENGTH,
    padding    = 'post',
    truncating = 'post',
    value      = 0,
    dtype      = 'int32'
)
y = df['label'].values.astype('int32')

print(f'\n📐 X : {X.shape}  dtype={X.dtype}')
print(f'📐 y : {y.shape}  dtype={y.dtype}')
print(f'\nExample')
print(f'  URL     : {df["url"].iloc[0]}')
print(f'  Encoded : {X[0][:40]}…')
print(f'  Label   : {y[0]}')
print('\n✅ Cell 4 complete.')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 5 — STRATIFIED TRAIN / TEST SPLIT  (80 / 20)
# ════════════════════════════════════════════════════════════

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size    = 0.20,
    random_state = SEED,
    stratify     = y
)

def class_balance(arr, name):
    u, c = np.unique(arr, return_counts=True)
    print(f'  {name}:')
    for cls, cnt in zip(u, c):
        tag = 'Benign  ' if cls == 0 else 'Phishing'
        print(f'    {tag} ({cls}): {cnt:>10,}  ({100*cnt/len(arr):.1f}%)')

print('═'*45)
print('       📊  SPLIT STATISTICS')
print('═'*45)
print(f'  Train : {len(X_train):,}   Test : {len(X_test):,}')
class_balance(y_train, 'Train')
class_balance(y_test,  'Test')
print('═'*45)
print('\n✅ Cell 5 complete.')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 6 — GPU-OPTIMISED MODEL ARCHITECTURE
#
# Larger than v1 to exploit ~15 GB GPU memory:
#   Embedding dim  : 128  (was 64)
#   Conv1 filters  : 256  (was 128)
#   Conv2 filters  : 128  (was 64)
#   Extra Conv3    : 64   (new)
#   Dense head     : 128 → 64
#   Dropout        : 0.4
# Output dtype cast to float32 for mixed-precision compatibility.
# ════════════════════════════════════════════════════════════

EMBEDDING_DIM = 128

def build_model(vocab_size: int, max_len: int, emb_dim: int = EMBEDDING_DIM) -> keras.Model:

    inp = keras.Input(shape=(max_len,), dtype='int32', name='url_chars')

    # ── Embedding ──────────────────────────────────────────
    x = layers.Embedding(
        input_dim  = vocab_size,
        output_dim = emb_dim,
        mask_zero  = False,
        name       = 'embedding'
    )(inp)

    # ── Conv Block 1  (wide receptive field) ───────────────
    x = layers.Conv1D(256, kernel_size=7, padding='same',
                      activation='relu', name='conv1')(x)
    x = layers.BatchNormalization(name='bn1')(x)
    x = layers.MaxPooling1D(pool_size=2, name='pool1')(x)

    # ── Conv Block 2  (medium receptive field) ─────────────
    x = layers.Conv1D(128, kernel_size=5, padding='same',
                      activation='relu', name='conv2')(x)
    x = layers.BatchNormalization(name='bn2')(x)
    x = layers.MaxPooling1D(pool_size=2, name='pool2')(x)

    # ── Conv Block 3  (fine-grained features) ──────────────
    x = layers.Conv1D(64, kernel_size=3, padding='same',
                      activation='relu', name='conv3')(x)
    x = layers.BatchNormalization(name='bn3')(x)

    # ── Global aggregation ─────────────────────────────────
    x = layers.GlobalMaxPooling1D(name='global_pool')(x)

    # ── Classifier head ────────────────────────────────────
    x = layers.Dense(128, activation='relu', name='fc1')(x)
    x = layers.Dropout(0.4, name='drop1')(x)
    x = layers.Dense(64, activation='relu', name='fc2')(x)
    x = layers.Dropout(0.3, name='drop2')(x)

    # Cast to float32 before sigmoid (required for mixed_float16)
    x = layers.Dense(1, name='logit')(x)
    out = layers.Activation('sigmoid', dtype='float32', name='output')(x)

    model = keras.Model(inputs=inp, outputs=out, name='CharCNN_v2')
    model.compile(
        optimizer = keras.optimizers.Adam(learning_rate=1e-3),
        loss      = 'binary_crossentropy',
        metrics   = ['accuracy', keras.metrics.AUC(name='auc')]
    )
    return model


model = build_model(VOCAB_SIZE, MAX_LENGTH)
model.summary(line_length=70)

total_params = model.count_params()
print(f'\n🔢 Total trainable parameters: {total_params:,}')

try:
    keras.utils.plot_model(
        model,
        to_file        = os.path.join(WORK_DIR, 'model_architecture.png'),
        show_shapes    = True,
        show_layer_names = True,
        dpi            = 100
    )
    print('🖼️  Architecture diagram saved.')
except Exception:
    pass

print('\n✅ Cell 6 complete.')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 7 — TRAINING CALLBACKS
# ════════════════════════════════════════════════════════════

BEST_MODEL_PATH = os.path.join(WORK_DIR, 'best_character_model.keras')
TB_LOG_DIR      = os.path.join(WORK_DIR, 'tb_logs')
os.makedirs(TB_LOG_DIR, exist_ok=True)

cb_early = callbacks.EarlyStopping(
    monitor              = 'val_loss',
    patience             = 7,          # slightly more generous for deep model
    restore_best_weights = True,
    verbose              = 1
)

cb_ckpt = callbacks.ModelCheckpoint(
    filepath      = BEST_MODEL_PATH,
    monitor       = 'val_loss',
    save_best_only= True,
    save_format   = 'keras',
    verbose       = 1
)

cb_lr = callbacks.ReduceLROnPlateau(
    monitor  = 'val_loss',
    factor   = 0.3,
    patience = 3,
    min_lr   = 1e-7,
    verbose  = 1
)

cb_tb = callbacks.TensorBoard(
    log_dir        = TB_LOG_DIR,
    histogram_freq = 1,
    update_freq    = 'epoch'
)

CALLBACKS = [cb_early, cb_ckpt, cb_lr, cb_tb]

print('🔧 Callbacks ready:')
for cb in CALLBACKS:
    print(f'   • {cb.__class__.__name__}')
print(f'\n💾 Best model path  : {BEST_MODEL_PATH}')
print('\n✅ Cell 7 complete.')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 8 — GPU-OPTIMISED TRAINING
#
# batch_size = 512  (utilises ~15 GB VRAM; lower if OOM)
# epochs     = 50  (EarlyStopping will halt before if needed)
# val_split  = 0.10
# ════════════════════════════════════════════════════════════

EPOCHS     = 50
BATCH_SIZE = 512   # ← tune down to 256 if you get OOM errors
VAL_SPLIT  = 0.10

print(f'🚀 Training config')
print(f'   Epochs      : {EPOCHS}  (EarlyStopping patience=7)')
print(f'   Batch size  : {BATCH_SIZE}')
print(f'   Val split   : {VAL_SPLIT*100:.0f}% of training set')
print(f'   Train rows  : {len(X_train):,}')
print(f'   Val rows    : ~{int(len(X_train)*VAL_SPLIT):,}')
print('─'*55)

history = model.fit(
    X_train, y_train,
    epochs           = EPOCHS,
    batch_size       = BATCH_SIZE,
    validation_split = VAL_SPLIT,
    callbacks        = CALLBACKS,
    verbose          = 1
)

n_epochs_ran = len(history.history['loss'])
print(f'\n🏁 Training finished after {n_epochs_ran} epoch(s).')
print(f'   Best val_loss : {min(history.history["val_loss"]):.4f}')
print(f'   Best val_auc  : {max(history.history["val_auc"]):.4f}')
print('\n✅ Cell 8 complete.')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 9 — FULL EVALUATION DASHBOARD
# ════════════════════════════════════════════════════════════

# ── Test-set metrics ─────────────────────────────────────────
print('⏳ Running evaluation on test set…')
test_loss, test_acc, test_auc_metric = model.evaluate(
    X_test, y_test, batch_size=512, verbose=0
)

y_prob = model.predict(X_test, batch_size=512, verbose=0).ravel()
y_pred = (y_prob >= 0.5).astype(int)
roc_auc = roc_auc_score(y_test, y_prob)

print(f'\n🎯 Test Loss     : {test_loss:.4f}')
print(f'🎯 Test Accuracy : {test_acc:.4f}')
print(f'🎯 Test AUC      : {roc_auc:.4f}')
print('\n' + '═'*55)
print('           📋  CLASSIFICATION REPORT')
print('═'*55)
print(classification_report(
    y_test, y_pred,
    target_names=['Benign (0)', 'Phishing (1)'],
    digits=4
))

# ── 2 × 2 Dashboard ──────────────────────────────────────────
fig = plt.figure(figsize=(15, 11))
fig.suptitle('Model Evaluation Dashboard – CharCNN v2', fontsize=16, fontweight='bold', y=1.01)
gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.35, wspace=0.30)

hist = history.history
ep   = range(1, len(hist['loss']) + 1)

# Loss
ax = fig.add_subplot(gs[0, 0])
ax.plot(ep, hist['loss'],     lw=2, label='Train',      color='#1f77b4')
ax.plot(ep, hist['val_loss'], lw=2, label='Validation', color='#ff7f0e', ls='--')
ax.set_title('Loss'); ax.set_xlabel('Epoch'); ax.set_ylabel('BCE Loss')
ax.legend(); ax.grid(alpha=0.3)

# Accuracy
ax = fig.add_subplot(gs[0, 1])
ax.plot(ep, hist['accuracy'],     lw=2, label='Train',      color='#2ca02c')
ax.plot(ep, hist['val_accuracy'], lw=2, label='Validation', color='#d62728', ls='--')
ax.set_title('Accuracy'); ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy')
ax.legend(); ax.grid(alpha=0.3)

# Confusion Matrix
ax = fig.add_subplot(gs[1, 0])
cm = confusion_matrix(y_test, y_pred)
cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100
annot  = np.array([[f'{v}\n({p:.1f}%)' for v, p in zip(row_v, row_p)]
                   for row_v, row_p in zip(cm, cm_pct)])
sns.heatmap(
    cm, annot=annot, fmt='', cmap='Blues',
    xticklabels=['Benign', 'Phishing'],
    yticklabels=['Benign', 'Phishing'],
    ax=ax, linewidths=0.5, cbar=False
)
ax.set_title('Confusion Matrix'); ax.set_xlabel('Predicted'); ax.set_ylabel('True')

# ROC Curve
ax = fig.add_subplot(gs[1, 1])
fpr, tpr, _ = roc_curve(y_test, y_prob)
ax.plot(fpr, tpr, lw=2.5, color='darkorange',
        label=f'AUC = {roc_auc:.4f}')
ax.plot([0,1],[0,1], 'k--', lw=1)
ax.fill_between(fpr, tpr, alpha=0.12, color='darkorange')
ax.set_xlim([0,1]); ax.set_ylim([0,1.02])
ax.set_title('ROC Curve'); ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
ax.legend(loc='lower right'); ax.grid(alpha=0.3)

plt.savefig(os.path.join(WORK_DIR, 'evaluation_dashboard.png'), dpi=150, bbox_inches='tight')
plt.show()
print('\n📊 Dashboard saved to Drive.')
print('\n✅ Cell 9 complete.')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 10 — MODEL & ARTIFACT EXPORT
# ════════════════════════════════════════════════════════════

FINAL_MODEL_PATH = os.path.join(WORK_DIR, 'final_model.keras')
model.save(FINAL_MODEL_PATH)
print(f'💾 Final model  → {FINAL_MODEL_PATH}')

# Refresh artifacts with embedding_dim used
ARTIFACTS['embedding_dim'] = EMBEDDING_DIM
ARTIFACTS['batch_size']    = BATCH_SIZE
joblib.dump(ARTIFACTS, ARTIFACTS_PATH)
print(f'💾 Artifacts    → {ARTIFACTS_PATH}')

# Manifest
print('\n📦 Export Manifest:')
files = [
    'final_model.keras',
    'best_character_model.keras',
    'preprocessing_artifacts.joblib',
    'evaluation_dashboard.png',
    'class_distribution.png',
]
for fname in files:
    p = os.path.join(WORK_DIR, fname)
    if os.path.exists(p):
        kb = os.path.getsize(p) / 1024
        print(f'   ✅  {fname:<45} {kb:>8.1f} KB')
    else:
        print(f'   ❌  {fname:<45}  MISSING')

print('\n✅ Cell 10 complete.')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 11 — INFERENCE HELPER FUNCTIONS
#
# Loads model + artifacts fresh from Drive.
# predict_url()  → single URL
# predict_batch()→ list of URLs  (returns DataFrame)
# ════════════════════════════════════════════════════════════

from tensorflow.keras.preprocessing.sequence import pad_sequences as _pad

# ── Load once (shared by Cell 12 too) ───────────────────────
_arts    = joblib.load(ARTIFACTS_PATH)
_c2i     = _arts['char2idx']
_maxlen  = _arts['max_length']
_model   = tf.keras.models.load_model(FINAL_MODEL_PATH)
print('✅ Inference engine loaded.')
print(f'   Vocab size  : {_arts["vocab_size"]}')
print(f'   Max length  : {_maxlen}')


def _preprocess(url: str) -> np.ndarray:
    """Clean → encode → pad a single URL string."""
    clean = str(url).lower().strip()
    seq   = [_c2i.get(ch, 0) for ch in clean[:_maxlen]]
    return _pad([seq], maxlen=_maxlen, padding='post',
                truncating='post', value=0, dtype='int32')


def predict_url(url: str, threshold: float = 0.5) -> dict:
    """
    Predict phishing probability for a single URL.

    Returns dict:
        url              – input URL
        phishing_prob    – float in [0, 1]
        predicted_label  – 0 or 1
        verdict          – 'Benign' | 'Phishing'
    """
    prob    = float(_model.predict(_preprocess(url), verbose=0)[0][0])
    label   = int(prob >= threshold)
    verdict = 'Phishing' if label else 'Benign'
    return dict(url=url, phishing_prob=round(prob, 6),
                predicted_label=label, verdict=verdict)


def predict_batch(urls: list, threshold: float = 0.5) -> pd.DataFrame:
    """Predict phishing probability for a list of URLs; returns a DataFrame."""
    Xb     = np.vstack([_preprocess(u) for u in urls])
    probs  = _model.predict(Xb, batch_size=256, verbose=0).ravel()
    labels = (probs >= threshold).astype(int)
    return pd.DataFrame({
        'url'            : urls,
        'phishing_prob'  : np.round(probs, 6),
        'predicted_label': labels,
        'verdict'        : ['Phishing' if l else 'Benign' for l in labels]
    })


print('\n✅ Cell 11 complete – helper functions ready.')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 12 — URL TESTING INTERFACE
#
# ┌─────────────────────────────────────────────────────────┐
# │  HOW TO USE                                             │
# │  ─────────                                              │
# │  Option A – single URL:                                │
# │      Set TEST_URL to any URL string and run cell.      │
# │                                                        │
# │  Option B – batch test:                                │
# │      Add URLs to TEST_URLS list and run cell.          │
# └─────────────────────────────────────────────────────────┘
# ════════════════════════════════════════════════════════════

# ╔══════════════════════════════════════════════════════════╗
# ║         ✏️  EDIT URL(s) HERE BEFORE RUNNING             ║
# ╚══════════════════════════════════════════════════════════╝

# ── Option A: single URL ─────────────────────────────────────
TEST_URL = "http://paypal-secure-login-update.xyz"

# ── Option B: batch of URLs ──────────────────────────────────
TEST_URLS = [
    # ── expected BENIGN ──────────────────────────────────────
    "https://www.google.com",
    "https://github.com/tensorflow/tensorflow",
    "https://www.microsoft.com/en-us/windows",
    "https://www.bbc.com/news",
    "https://stackoverflow.com/questions/tagged/python",
    # ── expected PHISHING ────────────────────────────────────
    "http://paypal-secure-login-update.xyz",
    "http://secure-login-update-paypal.xyz/verify?token=abc123",
    "http://paypal-account-suspended.com/login",
    "http://192.168.1.1/bankofamerica-login-update.html",
    "http://amazon-prize-winner-claim-now.ru/free-gift",
    "http://apple-id-locked-verify.tk/unlock",
]

THRESHOLD = 0.5   # decision boundary – raise for stricter phishing detection


# ──────────────────────────────────────────────────────────────
#  SINGLE URL RESULT
# ──────────────────────────────────────────────────────────────
def _prob_bar(prob: float, width: int = 35) -> str:
    filled = int(prob * width)
    color  = '🟥' if prob >= THRESHOLD else '🟩'
    return color * filled + '⬜' * (width - filled)

single = predict_url(TEST_URL, threshold=THRESHOLD)

verdict_icon = '🚨' if single['predicted_label'] == 1 else '✅'

print('\n' + '═'*58)
print('   🔍  SINGLE URL ANALYSIS')
print('═'*58)
print(f'  URL                  : {single["url"]}')
print(f'  Phishing Probability : {single["phishing_prob"]:.4f}')
print(f'  Probability Bar      : {_prob_bar(single["phishing_prob"])}')
print(f'  Prediction           : {verdict_icon}  {single["verdict"].upper()}')
print('═'*58)


# ──────────────────────────────────────────────────────────────
#  BATCH RESULTS TABLE
# ──────────────────────────────────────────────────────────────
print('\n\n' + '═'*58)
print('   📋  BATCH URL ANALYSIS')
print('═'*58)

results_df = predict_batch(TEST_URLS, threshold=THRESHOLD)

# Pretty-print each result
for _, row in results_df.iterrows():
    icon  = '🚨' if row['predicted_label'] == 1 else '✅'
    short = row['url'][:60]
    print(f"  {icon}  {row['phishing_prob']:.4f}  {row['verdict']:<9}  {short}")

print('═'*58)
n_phish_found = results_df['predicted_label'].sum()
print(f'  Scanned : {len(results_df)} URL(s) | '
      f'🚨 Phishing: {n_phish_found} | '
      f'✅ Benign: {len(results_df)-n_phish_found}')
print('═'*58)


# ──────────────────────────────────────────────────────────────
#  VISUAL PROBABILITY CHART
# ──────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, max(4, len(TEST_URLS)*0.55)))
colors  = ['#F44336' if l == 1 else '#4CAF50' for l in results_df['predicted_label']]
labels  = [f"{u[:50]}…" if len(u)>50 else u for u in results_df['url']]

bars = ax.barh(range(len(results_df)), results_df['phishing_prob'],
               color=colors, edgecolor='black', linewidth=0.5, height=0.6)
ax.axvline(THRESHOLD, color='black', linestyle='--', linewidth=1.5,
           label=f'Threshold = {THRESHOLD}')
ax.set_yticks(range(len(results_df)))
ax.set_yticklabels(labels, fontsize=8)
ax.set_xlim(0, 1)
ax.set_xlabel('Phishing Probability')
ax.set_title('URL Phishing Probability — Batch Results', fontsize=13, fontweight='bold')
ax.legend(loc='lower right')

# Annotate bars with probability values
for i, (bar, prob) in enumerate(zip(bars, results_df['phishing_prob'])):
    ax.text(min(prob + 0.01, 0.97), i, f'{prob:.3f}',
            va='center', fontsize=7.5, color='black')

# Legend patches
from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(facecolor='#F44336', label='Phishing'),
    Patch(facecolor='#4CAF50', label='Benign'),
], loc='lower right')

plt.tight_layout()
chart_path = os.path.join(WORK_DIR, 'batch_prediction_chart.png')
plt.savefig(chart_path, dpi=130, bbox_inches='tight')
plt.show()
print(f'\n📊 Batch chart saved → {chart_path}')
print('\n✅ Cell 12 complete – URL Testing Interface done.')